In [1]:
import torch
from torch import nn
from kerops.ops.conv import Conv3d, Conv3dWgrad

In [7]:
x = torch.randn(1, 16, 32, 32, 32, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
w = torch.randn(3, 3, 3, 16, 16, device='cuda', dtype=torch.float16)

Conv3d(x, weight=w)

tensor([[[[[-4.2114e-02,  7.2422e+00,  2.6895e+00,  ...,  2.0875e+01,
             7.1240e-01,  7.4375e+00],
           [-4.5781e+00,  1.9891e+01,  4.2031e+00,  ..., -1.4629e+00,
            -9.0625e+00,  1.7875e+01],
           [-2.7953e+01,  1.8688e+01,  1.0383e+01,  ...,  3.2781e+01,
             1.2803e+00,  7.7969e+00],
           ...,
           [ 1.4242e+01, -2.9258e+00,  2.8812e+01,  ...,  1.7734e+01,
            -1.1133e+01,  5.0391e+00],
           [-1.3773e+01, -2.2500e+01,  3.3320e+00,  ...,  2.0375e+01,
            -1.3836e+01, -6.8594e+00],
           [ 8.6953e+00, -2.4156e+01, -1.6016e+01,  ...,  9.2578e+00,
             9.8594e+00,  1.1273e+01]],

          [[-5.1211e+00, -1.4078e+01,  4.1906e+01,  ..., -2.0859e+00,
            -5.6016e+00, -8.4141e+00],
           [ 5.0234e+00,  2.5922e+01,  7.4180e+00,  ...,  1.0398e+01,
             1.3266e+01, -7.2773e+00],
           [ 1.1500e+01, -2.5000e+01,  3.0098e+00,  ..., -1.1514e+00,
             1.2719e+01, -1.7922e+01],
 

In [2]:
import itertools
import json
from tqdm.notebook import tqdm
from time import perf_counter, sleep

import numpy as np
from joblib import Parallel, delayed


def mean_std_percentile(x, lo=20, hi=80):
    x = np.asarray(x, dtype=np.float32)

    p_lo, p_hi = np.percentile(x, [lo, hi])

    mask = (x >= p_lo) & (x <= p_hi)
    x_mid = x[mask]

    return x_mid.mean(), x_mid.std()


def bench(func, *args, n_jobs_precompile=4, warmup=25, sleep_ms=100, n_iters=50, q_show=10, quantiles=(20, 80), savefile=None, **specset):
    results = []

    keys = list(specset.keys())
    values = list(specset.values())
    configs = list(itertools.product(*values))

    def precompile_call(config):
        kwargs = dict(zip(keys, config))
        func(*args, **kwargs)

    n_jobs_precompile = min(n_jobs_precompile, len(configs))
    Parallel(n_jobs=n_jobs_precompile, backend='threading')(delayed(precompile_call)(config) for config in tqdm(configs, desc="Precompiling"))

    for config in tqdm(configs, desc="Benchmark configs"):
        if sleep_ms is not None:
            sleep(sleep_ms / 1000)
        kwargs = dict(zip(keys, config))

        try:
            func(*args, **kwargs)
            torch.cuda.synchronize()
        except Exception as e:
            pass

        for _ in range(warmup):
            func(*args, **kwargs)
        torch.cuda.synchronize()

        times_ms = []

        for _ in range(n_iters):
            start = perf_counter()
            func(*args, **kwargs)
            torch.cuda.synchronize()
            end = perf_counter()
            times_ms.append((end - start) * 1e3)

        mean, std = mean_std_percentile(times_ms, *quantiles)
        results.append({
            "spec": kwargs,
            "mean_ms": float(mean),
            "std_ms": float(std),
        })

    best_result = min(results, key=lambda x: x["mean_ms"])
    best_mean = best_result['mean_ms']
    best_std = best_result['std_ms']
    best_spec = best_result['spec']
    print(f'Best spec - {best_mean:.3f}+-{best_std:.3f}ms {best_spec}')

    good_results = []
    for result in results:
        if best_mean * (1 + q_show / 100) >= result["mean_ms"] and result['spec'] != best_spec:
            good_results.append(result)

    if good_results:
        print(30 * '-')
        print("Other good specs:")

        for result in good_results:
            print(f'{result['mean_ms']:.3f}+-{result['std_ms']:.3f}ms {result['spec']}')
    else:
        print(f'Other specs have a time difference of more than {q_show}%')

    if savefile is not None:
        with open(savefile, 'w') as f:
            json.dump({'best': best_result, 'good_results': good_results}, f)

In [ ]:
from functools import partial
from kerops.ops.conv.conv import num_warps, d_block, cin_block


cudnn_conv = partial(nn.functional.conv3d, stride=1, padding=1)


channels = [16, 32, 64, 128]


def get_size(CIN, COUT):
    if CIN <= 32 and COUT <= 32:
        return 128
    elif CIN <= 64 and COUT <= 64:
        return 96
    else:
        return 64


for CIN in channels:
    for COUT in channels:
        if CIN == COUT or 2 * CIN == COUT or CIN == 2 * COUT:
            print(f'{CIN=} {COUT=}')

            BS = 1
            S = get_size(CIN, COUT)
            D = get_size(CIN, COUT)

            x = torch.randn(BS, CIN, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
            grad = torch.randn(BS, COUT, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
            w = torch.randn(3, 3, 3, CIN, COUT, device='cuda', dtype=torch.float16)
            weight = w.permute(-1, -2, 0, 1, 2).contiguous()

            print('default')
            bench(Conv3d, x, w)
            print('cudnn')
            bench(cudnn_conv, x, weight, None)
            print('bench')
            CIN_BLOCKS = [16, 32] if CIN >= 32 else [16]
            D_BLOCKS = [16, 32, 64] if (CIN <= 32 and COUT <= 32) else [16, 32]
            bench(Conv3d, x, w, savefile=f'Conv3dV6_{CIN}_{COUT}.json', num_warps=[1, 2, 4], D_BLOCK=D_BLOCKS, CIN_BLOCK=CIN_BLOCKS, LOAD_WEIGHT_FIRST=[True, False], WEIGHT_MAJOR=[False, True])
            
            print(30 * '-')

### Conv3dWgradV1

In [69]:
import triton
from triton import language as tl, next_power_of_2


@triton.jit
def _Conv_wgrad_cl3d_impl_V1(
    grad_ptr,
    input_ptr,
    weight_grad_ptr,
    H,
    W,
    D,
    num_buffers,
    ACCTYPE: tl.constexpr,
    D_BLOCK: tl.constexpr,
    IN_CHANNELS: tl.constexpr,
    OUT_CHANNELS: tl.constexpr,
    CIN_BLOCK: tl.constexpr,
    COUT_BLOCK: tl.constexpr,
):
    W_pid = tl.program_id(0)
    H_pid = tl.program_id(1)
    D_pid = tl.program_id(2)

    CIN_STEPS: tl.constexpr = IN_CHANNELS // CIN_BLOCK
    COUT_STEPS: tl.constexpr = OUT_CHANNELS // COUT_BLOCK

    linear = W_pid + H_pid * tl.num_programs(0) + D_pid * tl.num_programs(0) * tl.num_programs(1)
    buffer_idx = linear % num_buffers
    
    in_channels_offset = tl.arange(0, CIN_BLOCK)
    out_channels_offset = tl.arange(0, COUT_BLOCK)
    d_offset = tl.arange(0, D_BLOCK)

    grad_offset = d_offset[:, None] * OUT_CHANNELS + out_channels_offset[None, :]
    input_offset = d_offset[None, :] * IN_CHANNELS + in_channels_offset[:, None]
    weight_grad_offset = in_channels_offset[:, None] * OUT_CHANNELS + out_channels_offset[None, :]

    grad_ptr += D_pid * D_BLOCK * OUT_CHANNELS
    grad_ptr += W_pid * D * OUT_CHANNELS
    grad_ptr += H_pid * D * W * OUT_CHANNELS

    input_ptr += D_pid * D_BLOCK * IN_CHANNELS
    input_ptr += W_pid * D * IN_CHANNELS
    input_ptr += H_pid * D * W * IN_CHANNELS

    for cout in tl.static_range(0, COUT_STEPS):
        grad = tl.load(grad_ptr + cout * COUT_BLOCK + grad_offset, other=0, mask=(d_offset < (D - D_pid * D_BLOCK))[:, None])
        
        for cin in tl.static_range(0, CIN_STEPS):
            for h in tl.static_range(-1, 2):
                for w in tl.static_range(-1, 2):
                    for d in tl.static_range(-1, 2):
                        x_ptr = (
                            input_ptr
                            + h * IN_CHANNELS * D * W
                            + w * IN_CHANNELS * D
                            + d * IN_CHANNELS
                            + cin * CIN_BLOCK
                        )
        
                        mask = (d_offset < (D - D_pid * D_BLOCK - d))[None, :] & (d_offset >= (- D_pid * D_BLOCK - d))[None, :] & ((H_pid + h) < H) & ((H_pid + h) >= 0) & ((W_pid + w) < W) & ((W_pid + w) >= 0)
        
                        x = tl.load(x_ptr + input_offset, other=0, mask=mask)
        
                        wgrad = tl.dot(x, grad, out_dtype=ACCTYPE)
        
                        w_ptr = (
                            weight_grad_ptr
                            + buffer_idx * IN_CHANNELS * OUT_CHANNELS * 3 * 3 * 3
                            + (h + 1) * IN_CHANNELS * OUT_CHANNELS * 3 * 3
                            + (w + 1) * IN_CHANNELS * OUT_CHANNELS * 3
                            + (d + 1) * IN_CHANNELS * OUT_CHANNELS
                            + cin * CIN_BLOCK * OUT_CHANNELS
                            + cout * COUT_BLOCK
                        )
                        tl.atomic_add(w_ptr + weight_grad_offset, wgrad, sem='relaxed')

In [4]:
from kerops.utils import cdiv


def Conv3dWgradV1(grad, x, D_BLOCK, ACCTYPE, num_warps, REDUCTION_FACTOR, CIN_BLOCK, COUT_BLOCK):
    assert x.device == grad.device
    assert x.is_cuda

    assert x.ndim == grad.ndim == 5
    xbsize, in_channels, xH, xW, xD = x.shape
    gbsize, out_channels, gH, gW, gD = grad.shape
    assert in_channels == next_power_of_2(in_channels)
    assert out_channels == next_power_of_2(out_channels)
    assert [xbsize, xH, xW, xD] == [gbsize, gH, gW, gD]

    assert xbsize == 1  # TODO: batched version

    assert x.is_contiguous(memory_format=torch.channels_last_3d)
    assert grad.is_contiguous(memory_format=torch.channels_last_3d)

    assert x.dtype == grad.dtype == torch.float16

    assert D_BLOCK == next_power_of_2(D_BLOCK)
    assert ACCTYPE in ('float16', 'float32')
    assert CIN_BLOCK == next_power_of_2(CIN_BLOCK)
    assert CIN_BLOCK <= in_channels
    assert COUT_BLOCK == next_power_of_2(COUT_BLOCK)
    assert COUT_BLOCK <= out_channels
    assert isinstance(REDUCTION_FACTOR, int) or REDUCTION_FACTOR == None

    ACCTYPE = {'float32': tl.float32, 'float16': tl.float16}[ACCTYPE]
    num_buffers = 1 if REDUCTION_FACTOR is None else cdiv(xW, REDUCTION_FACTOR) * cdiv(xH, REDUCTION_FACTOR) * cdiv(D, D_BLOCK * REDUCTION_FACTOR)
    weight_grad = torch.zeros([num_buffers, 3, 3, 3, in_channels, out_channels], device=x.device, dtype=torch.float16)
    grid = (xW, xH, cdiv(xD, D_BLOCK))

    _Conv_wgrad_cl3d_impl_V1[grid](
        grad,
        x,
        weight_grad,
        xH,
        xW,
        xD,
        num_buffers,
        IN_CHANNELS=in_channels,
        OUT_CHANNELS=out_channels,
        ACCTYPE=ACCTYPE,
        D_BLOCK=D_BLOCK,
        CIN_BLOCK=CIN_BLOCK,
        COUT_BLOCK=COUT_BLOCK,
        num_warps=num_warps,
    )

    return weight_grad.sum(dim=0)

### Conv3dWgradV2

In [150]:
import triton
from triton import language as tl, next_power_of_2


@triton.jit
def _Conv_wgrad_cl3d_impl_V2(
    grad_ptr,
    input_ptr,
    weight_grad_ptr,
    H,
    W,
    D,
    num_buffers,
    ACCTYPE: tl.constexpr,
    D_BLOCK: tl.constexpr,
    IN_CHANNELS: tl.constexpr,
    OUT_CHANNELS: tl.constexpr,
    CIN_BLOCK: tl.constexpr,
    COUT_BLOCK: tl.constexpr,
):
    WCOUT_pid = tl.program_id(0)
    H_pid = tl.program_id(1)
    BD_pid = tl.program_id(2)

    W_pid = WCOUT_pid // tl.cdiv(OUT_CHANNELS, COUT_BLOCK)
    COUT_pid = WCOUT_pid % tl.cdiv(OUT_CHANNELS, COUT_BLOCK)

    B_pid = BD_pid // tl.cdiv(D, D_BLOCK)
    D_pid = BD_pid % tl.cdiv(D, D_BLOCK)

    CIN_STEPS: tl.constexpr = IN_CHANNELS // CIN_BLOCK

    linear = W_pid + H_pid * tl.num_programs(0) + BD_pid * tl.num_programs(0) * tl.num_programs(1)
    buffer_idx = linear % num_buffers
    
    in_channels_offset = tl.arange(0, CIN_BLOCK)
    out_channels_offset = tl.arange(0, COUT_BLOCK)
    d_offset = tl.arange(0, D_BLOCK)

    grad_offset = d_offset[:, None] * OUT_CHANNELS + out_channels_offset[None, :]
    input_offset = d_offset[None, :] * IN_CHANNELS + in_channels_offset[:, None]
    weight_grad_offset = in_channels_offset[:, None] * OUT_CHANNELS + out_channels_offset[None, :]

    grad_ptr += COUT_pid * COUT_BLOCK
    grad_ptr += D_pid * D_BLOCK * OUT_CHANNELS
    grad_ptr += W_pid * 2 * D * OUT_CHANNELS
    grad_ptr += H_pid * 2 * D * W * OUT_CHANNELS
    grad_ptr += B_pid * H * W * D * OUT_CHANNELS

    input_ptr += D_pid * D_BLOCK * IN_CHANNELS
    input_ptr += W_pid * 2 * D * IN_CHANNELS
    input_ptr += H_pid * 2 * D * W * IN_CHANNELS
    input_ptr += B_pid * H * W * D * IN_CHANNELS

    weight_grad_ptr += COUT_pid * COUT_BLOCK

    g01 = ((W_pid * 2 + 1) < W)
    g10 = ((H_pid * 2 + 1) < H)
    g11 = ((W_pid * 2 + 1) < W) & ((H_pid * 2 + 1) < H)

    gmask = (d_offset < (D - D_pid * D_BLOCK))
    gmask = gmask[:, None]

    grads = [
        [
            tl.load(grad_ptr + grad_offset, other=0, mask=gmask),
            tl.load(grad_ptr + grad_offset + OUT_CHANNELS * D, other=0, mask=gmask & g01),
        ],
        [
            tl.load(grad_ptr + grad_offset + OUT_CHANNELS * D * W, other=0, mask=gmask & g10),
            tl.load(grad_ptr + grad_offset + OUT_CHANNELS * D * W + OUT_CHANNELS * D, other=0, mask=gmask & g11)
        ]
    ]

    for h in tl.static_range(-1, 2):
        for w in tl.static_range(-1, 2):
            for d in tl.static_range(-1, 2):
                for cin in tl.static_range(0, CIN_STEPS):
                    wgrad = tl.zeros([CIN_BLOCK, COUT_BLOCK], dtype=ACCTYPE)
                    
                    x_ptr = (
                        input_ptr
                        + h * IN_CHANNELS * D * W
                        + w * IN_CHANNELS * D
                        + d * IN_CHANNELS
                        + cin * CIN_BLOCK
                    )
    
                    xmask = (d_offset < (D - D_pid * D_BLOCK - d)) & (d_offset >= (- D_pid * D_BLOCK - d))
                    xmask = xmask[None, :]

                    x00 = ((H_pid * 2 + h) < H) & ((H_pid * 2 + h) >= 0) & ((W_pid * 2 + w) < W) & ((W_pid * 2 + w) >= 0)
                    x01 = ((H_pid * 2 + h) < H) & ((H_pid * 2 + h) >= 0) & ((W_pid * 2 + w + 1) < W) & ((W_pid * 2 + w + 1) >= 0)
                    x10 = ((H_pid * 2 + h + 1) < H) & ((H_pid * 2 + h + 1) >= 0) & ((W_pid * 2 + w) < W) & ((W_pid * 2 + w) >= 0)
                    x11 = ((H_pid * 2 + h + 1) < H) & ((H_pid * 2 + h + 1) >= 0) & ((W_pid * 2 + w + 1) < W) & ((W_pid * 2 + w + 1) >= 0)

                    xs = [
                        [
                            tl.load(x_ptr + input_offset, other=0, mask=xmask & x00),
                            tl.load(x_ptr + input_offset + IN_CHANNELS * D, other=0, mask=xmask & x01),
                        ],
                        [
                            tl.load(x_ptr + input_offset + IN_CHANNELS * D * W, other=0, mask=xmask & x10),
                            tl.load(x_ptr + input_offset + IN_CHANNELS * D * W + IN_CHANNELS * D, other=0, mask=xmask & x11),
                        ]
                    ]
    
                    for kh in tl.static_range(0, 2):
                        for kw in tl.static_range(0, 2):
                            wgrad += tl.dot(xs[kh][kw], grads[kh][kw])

    
                    w_ptr = (
                        weight_grad_ptr
                        + buffer_idx * IN_CHANNELS * OUT_CHANNELS * 3 * 3 * 3
                        + (h + 1) * IN_CHANNELS * OUT_CHANNELS * 3 * 3
                        + (w + 1) * IN_CHANNELS * OUT_CHANNELS * 3
                        + (d + 1) * IN_CHANNELS * OUT_CHANNELS
                        + cin * CIN_BLOCK * OUT_CHANNELS
                    )
                    tl.atomic_add(w_ptr + weight_grad_offset, wgrad, sem='relaxed')

In [151]:
from kerops.utils import cdiv


def Conv3dWgradV2(grad, x, D_BLOCK, ACCTYPE, num_warps, REDUCTION_FACTOR, CIN_BLOCK, COUT_BLOCK):
    assert x.device == grad.device
    assert x.is_cuda

    assert x.ndim == grad.ndim == 5
    xbsize, in_channels, xH, xW, xD = x.shape
    gbsize, out_channels, gH, gW, gD = grad.shape
    assert in_channels == next_power_of_2(in_channels)
    assert out_channels == next_power_of_2(out_channels)
    assert [xbsize, xH, xW, xD] == [gbsize, gH, gW, gD]

    assert x.is_contiguous(memory_format=torch.channels_last_3d)
    assert grad.is_contiguous(memory_format=torch.channels_last_3d)

    assert x.dtype == grad.dtype == torch.float16

    assert D_BLOCK == next_power_of_2(D_BLOCK)
    assert ACCTYPE in ('float16', 'float32')
    assert CIN_BLOCK == next_power_of_2(CIN_BLOCK)
    assert CIN_BLOCK <= in_channels
    assert isinstance(REDUCTION_FACTOR, int)

    ACCTYPE = {'float32': tl.float32, 'float16': tl.float16}[ACCTYPE]
    num_buffers = cdiv(xW, REDUCTION_FACTOR * 2) * cdiv(xH, REDUCTION_FACTOR * 2) * cdiv(xD, D_BLOCK * REDUCTION_FACTOR) * xbsize
    weight_grad = torch.zeros([num_buffers, 3, 3, 3, in_channels, out_channels], device=x.device, dtype=torch.float32)
    grid = (cdiv(xW, 2) * cdiv(out_channels, COUT_BLOCK), cdiv(xH, 2), cdiv(xD, D_BLOCK) * xbsize)

    _Conv_wgrad_cl3d_impl_V2[grid](
        grad,
        x,
        weight_grad,
        xH,
        xW,
        xD,
        num_buffers,
        IN_CHANNELS=in_channels,
        OUT_CHANNELS=out_channels,
        ACCTYPE=ACCTYPE,
        D_BLOCK=D_BLOCK,
        CIN_BLOCK=CIN_BLOCK,
        COUT_BLOCK=COUT_BLOCK,
        num_warps=num_warps,
    )

    return weight_grad.sum(dim=0).to(torch.float16)

In [3]:
from kerops.utils import weight_grad_similarity

S = 128
D = 128
CIN = 16
COUT = 16
BS = 1

x = torch.randn(BS, CIN, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
grad = torch.randn(BS, COUT, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
w = torch.randn(3, 3, 3, CIN, COUT, device='cuda', dtype=torch.float16)
weight = w.permute(-1, -2, 0, 1, 2).contiguous()


#grad_w = Conv3dWgradV2(grad, x, **{'D_BLOCK': 32, 'num_warps': 4, 'ACCTYPE': 'float32', 'REDUCTION_FACTOR': 32, 'CIN_BLOCK': 16, 'COUT_BLOCK': 64})
#grad_w = Conv3dWgradV2(grad, x, D_BLOCK=32, ACCTYPE='float32', num_warps=2, REDUCTION_FACTOR=32, CIN_BLOCK=32, COUT_BLOCK=32)
grad_w = Conv3dWgrad(grad, x)

_, grad_w_reference, _ = torch.ops.aten.convolution_backward(
    grad,
    x,
    weight,
    [0],  # bias_sizes
    [1, 1, 1],  # stride
    [1, 1, 1],  # padding
    [1, 1, 1],  # dilation
    False,  # transposed
    [0, 0, 0],  # output padding
    1,  # groups!
    [False, True, False],  # output_mask - grad_inpt, grad_weight, grad_bias
)

grad_w_p = grad_w.permute(-1, -2, 0, 1, 2)

weight_grad_similarity(grad_w_reference.float(), grad_w_p.float(), rtol_cos=0.0001, debug_info='print')

All stages passed


True

In [8]:
BS = 1
S = 128
D = 128
CIN = 64
COUT = 64

x = torch.randn(BS, CIN, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
grad = torch.randn(BS, COUT, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
w = torch.randn(3, 3, 3, CIN, COUT, device='cuda', dtype=torch.float16)
weight = w.permute(-1, -2, 0, 1, 2).contiguous()

In [51]:
channels = [16, 32, 64, 128]

with_fwd = True
with_bwd = True

for CIN in channels:
    for COUT in channels:
        if CIN == COUT or 2 * CIN == COUT or CIN == 2 * COUT:
            print(f'{CIN=} {COUT=}')
            BS = 1
            S = 128
            D = 128

            x = torch.randn(BS, CIN, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
            grad = torch.randn(BS, COUT, S, S, D, device='cuda', dtype=torch.float16).to(memory_format=torch.channels_last_3d)
            w = torch.randn(3, 3, 3, CIN, COUT, device='cuda', dtype=torch.float16)
            weight = w.permute(-1, -2, 0, 1, 2).contiguous()

            for _ in range(5):
                if with_fwd:
                    nn.functional.conv3d(x, weight, None, stride=1, padding=1)
                if with_bwd:
                    _, grad_w, _ = torch.ops.aten.convolution_backward(
                        grad,
                        x,
                        weight,
                        [0],  # bias_sizes
                        [1, 1, 1],  # stride
                        [1, 1, 1],  # padding
                        [1, 1, 1],  # dilation
                        False,  # transposed
                        [0, 0, 0],  # output padding
                        1,  # groups!
                        [False, True, False],  # output_mask - grad_inpt, grad_weight, grad_bias
                    )
            torch.cuda.synchronize()

            times_ms_cudnn = []
    
            for _ in range(25):
                start = perf_counter()
                if with_fwd:
                    nn.functional.conv3d(x, weight, None, stride=1, padding=1)
                if with_bwd:
                    torch.ops.aten.convolution_backward(
                        grad,
                        x,
                        weight,
                        [0],  # bias_sizes
                        [1, 1, 1],  # stride
                        [1, 1, 1],  # padding
                        [1, 1, 1],  # dilation
                        False,  # transposed
                        [0, 0, 0],  # output padding
                        1,  # groups!
                        [False, True, False],  # output_mask - grad_inpt, grad_weight, grad_bias
                    )
                torch.cuda.synchronize()
                end = perf_counter()
                times_ms_cudnn.append((end - start) * 1e3)

            for _ in range(5):
                if with_fwd:
                    Conv3d(x, w)
                if with_bwd:
                    Conv3dWgrad(grad, x)
            torch.cuda.synchronize()

            times_ms_triton = []
    
            for _ in range(25):
                start = perf_counter()
                if with_fwd:
                    Conv3d(x, w)
                if with_bwd:
                    Conv3dWgrad(grad, x)
                torch.cuda.synchronize()
                end = perf_counter()
                times_ms_triton.append((end - start) * 1e3)

            speedup = np.mean(times_ms_cudnn) / np.mean(times_ms_triton)

            print(f'Speedup - {speedup:.3f}')
                
            print(30 * '-')

CIN=16 COUT=16
Speedup - 1.748
------------------------------
CIN=16 COUT=32
Speedup - 1.207
------------------------------
CIN=32 COUT=16
Speedup - 1.690
------------------------------
CIN=32 COUT=32
Speedup - 1.112
------------------------------
CIN=32 COUT=64
Speedup - 1.074
------------------------------
CIN=64 COUT=32
Speedup - 1.136
------------------------------
CIN=64 COUT=64
Speedup - 0.952
------------------------------
CIN=64 COUT=128
Speedup - 1.003
------------------------------
CIN=128 COUT=64
Speedup - 0.951
------------------------------
CIN=128 COUT=128
Speedup - 0.960
------------------------------
